# 04 — Export checkpoints to Core ML

Converts the trained artifacts (notebooks 02 + 03) to fp16 `.mlpackage` for the iOS app. Conversion runs fine on Linux; runtime verification happens on a Mac/iPhone. No GPU needed.

Reads checkpoints from Drive, writes zipped `.mlpackage`s back to Drive (an `.mlpackage` is a directory).

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TORCH_HOME'] = '/content/torch-cache'

SEG_CKPT = f'{DRIVE_ROOT}/checkpoints/segformer-mit-b0-food'
REG_CKPT = f'{DRIVE_ROOT}/checkpoints/mass-regressor.pt'
OUT = f'{DRIVE_ROOT}/out'

# Clone the repo. Private repo → add your GitHub token as a Colab secret named
# GH_TOKEN (🔑 left sidebar). Public repo → no token needed. Fails LOUD.
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
repo = 'github.com/Hakeem-Hannoon/MacroScope-Portion-Scout.git'
url = f'https://x-access-token:{token}@{repo}' if token else f'https://{repo}'
# A prior run may leave the kernel's CWD inside /content/ppe (the %cd in the
# next cell). Deleting that dir out from under the kernel makes the git clone
# below fail with "Unable to read current working directory". Reset to a
# stable dir first so re-runs are safe.
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/ppe'])
r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/ppe'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/ppe/model'), (
    (r.stderr.replace(token, '***') if token else r.stderr) +
    '\nClone failed — add a GH_TOKEN Colab secret (🔑) or make the repo public.')
print('repo ready')

%pip -q install coremltools transformers timm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo ready


In [9]:
%cd /content/ppe
!python model/export/export_coreml.py segformer --checkpoint "{SEG_CKPT}" --out /content/FoodSeg.mlpackage
!python model/export/export_coreml.py regressor --checkpoint "{REG_CKPT}" --out /content/MassRegressor.mlpackage
!cd /content && zip -qr "{OUT}/FoodSeg.mlpackage.zip" FoodSeg.mlpackage && zip -qr "{OUT}/MassRegressor.mlpackage.zip" MassRegressor.mlpackage
!ls -lh "{OUT}" | grep mlpackage

/content/ppe
scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
XGBoost version 3.3.0 has not been tested with coremltools. You may run into unexpected errors. XGBoost 1.4.2 is the most recent version that has been tested.
2026-07-09 01:37:41.798345: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-09 01:37:42.126263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
TensorFlow version 2.20.0 has not been tested with coremltools. You may run into unexpected errors. TensorFlow 2.12.0 is the most recent version that has been tested.
Torch version 2.11.0+cpu has not been tested with co